# Day 074 — Exercise 3: frames_to_video

**What you'll build:** `frames_to_video(frames, output_path, fps, fourcc, writer_fn) -> Path` — write a list of numpy frame arrays to a video file.

**Why it matters:** Completing the read→process→write loop — every video processing pipeline ends by writing the result to disk.

In [ ]:
from pathlib import Path

def _make_test_frames(n=10, height=32, width=32):
    import numpy as np
    frames = []
    for i in range(n):
        frame = np.zeros((height, width, 3), dtype=np.uint8)
        frame[:, :, 0] = int(255 * i / max(n - 1, 1))
        frames.append(frame)
    return frames
_MOCK_META = {
    'fps': 30.0, 'frame_count': 10, 'width': 32, 'height': 32, 'duration_sec': 0.333,
}
_mock_info_fn    = lambda source: dict(_MOCK_META)
_mock_capture_fn = lambda source: _make_test_frames(10)
_mock_writer_fn  = lambda frames, path, fps: (Path(path).write_bytes(b'VIDEO' + bytes(len(frames))), Path(path))[1]
_mock_ffmpeg_fn  = lambda args: {'returncode': 0, 'stdout': '', 'stderr': ''}


## Task

- **Mock:** `return writer_fn(frames, output_path, fps)`
- **Real:** `import cv2`, `list(frames)`, raise `ValueError` if empty, `h,w = frames[0].shape[:2]`, `VideoWriter_fourcc(*fourcc)`, `VideoWriter(str(path), code, fps, (w,h))`, write loop, `release()`, return `Path`

**Key gotcha:** VideoWriter takes `(width, height)` — numpy shape is `(height, width, channels)` so pass `(w, h)` not `(h, w)`.

## Your Implementation

In [ ]:
def frames_to_video(frames: list, output_path,
                    fps: float = 30.0, fourcc: str = 'mp4v',
                    writer_fn=None):
    """Write frames to a video file.

    Args:
        frames:      list of (H, W, 3) uint8 numpy arrays (BGR)
        output_path: destination file path
        fps:         output frame rate
        fourcc:      codec code ('mp4v' for .mp4, 'XVID' for .avi)
        writer_fn:   callable(frames, output_path, fps) -> Path for testing
    Returns:
        Path to written file
    """
    raise NotImplementedError


In [ ]:
def frames_to_video(frames, output_path, fps=30.0, fourcc='mp4v', writer_fn=None):
    if writer_fn is not None:
        return writer_fn(frames, output_path, fps)
    import cv2
    frames = list(frames)
    if not frames:
        raise ValueError('frames list is empty')
    h, w        = frames[0].shape[:2]
    fourcc_code = cv2.VideoWriter_fourcc(*fourcc)
    out_path    = Path(output_path)
    writer      = cv2.VideoWriter(str(out_path), fourcc_code, fps, (w, h))
    for frame in frames:
        writer.write(frame)
    writer.release()
    return out_path


## Automated checks

In [ ]:

import tempfile
score, total = 0, 5
try:
    test_frames = _make_test_frames(6)

    # returns Path
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
        tmp = f.name
    out = frames_to_video(test_frames, tmp, writer_fn=_mock_writer_fn)
    assert isinstance(out, Path), f"expected Path, got {type(out)}"
    score += 1; print("✅ returns Path object")

    # file exists and has content
    assert out.exists() and out.stat().st_size > 0
    score += 1; print("✅ output file exists with non-zero size")

    # fps is forwarded to writer_fn
    captured_fps = {}
    def _cap_writer(frames, path, fps):
        captured_fps['fps'] = fps
        return Path(path)
    frames_to_video(test_frames, tmp, fps=24.0, writer_fn=_cap_writer)
    assert captured_fps.get('fps') == 24.0
    score += 1; print("✅ fps forwarded to writer_fn")

    # frames list forwarded correctly
    captured_frames = {}
    def _cap_frames(frames, path, fps):
        captured_frames['n'] = len(frames)
        return Path(path)
    frames_to_video(test_frames, tmp, writer_fn=_cap_frames)
    assert captured_frames.get('n') == 6
    score += 1; print("✅ correct number of frames forwarded")

    # different frame counts → different file sizes (mock encodes count)
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
        tmp2 = f.name
    out1 = frames_to_video(_make_test_frames(3), tmp, writer_fn=_mock_writer_fn)
    out2 = frames_to_video(_make_test_frames(8), tmp2, writer_fn=_mock_writer_fn)
    assert out1.stat().st_size != out2.stat().st_size
    score += 1; print("✅ different frame counts produce different file sizes")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def frames_to_video(frames, output_path, fps=30.0, fourcc='mp4v', writer_fn=None):
    if writer_fn is not None:
        return writer_fn(frames, output_path, fps)
    import cv2
    frames = list(frames)
    if not frames:
        raise ValueError('frames list is empty')
    h, w        = frames[0].shape[:2]
    fourcc_code = cv2.VideoWriter_fourcc(*fourcc)
    out_path    = Path(output_path)
    writer      = cv2.VideoWriter(str(out_path), fourcc_code, fps, (w, h))
    for frame in frames:
        writer.write(frame)
    writer.release()
    return out_path
```

**Why `list(frames)`?** The caller might pass a generator or iterator. Converting to list once means we can safely read `frames[0]` for shape and iterate in the write loop.

</details>